based on [tutorial](https://medium.com/@maneyogesh065/fine-tuning-biobert-for-custom-named-entity-recognition-a-complete-guide-a05b124edda0)

create custom aal dictionary

In [ ]:
#custom dict

alias

In [1]:
import json

with open("../../data/AAL3v1.json", "r", encoding="utf8") as f:
    aal = json.load(f)

In [ ]:
alias_to_label = {}

for region in aal.values():
    label = region["aal_name"]
    aliases = set()
    aliases.add(region["full_name"])
    aliases.update(region["synonyms"])

    for alias in aliases:
        alias_to_label[alias.lower()] = label

In [ ]:
def expand(alias):
    variants = {alias}
    variants.add(alias.replace("-", " "))
    variants.add(alias.replace("_", " "))
    variants.add(alias.replace("-", ""))

    return variants

In [ ]:
all_aliases = {}

for region in aal.values():
    label = region["aal_name"]
    names = [region["full_name"]] + region["synonyms"]
    for n in names:
        for variant in expand(n):
            all_aliases[variant.lower()] = label

In [8]:
import pandas as pd

df = pd.read_csv("../../data/abstracts.csv")

texts = df["0"].tolist()

look for brain region mentions in the abstracts

In [10]:
import re

def annotate(text):

    matches = []

    lower = text.lower()

    for alias, label in all_aliases.items():

        pattern = r"\b" + re.escape(alias) + r"\b"

        for m in re.finditer(pattern, lower):

            matches.append({
                "start": m.start(),
                "end": m.end(),
                "text": text[m.start():m.end()],
                "label": label
            })

    return sorted(matches, key=lambda x: x["start"])

In [16]:
annotate("the brain region that works the best is hippocampus left precentral gyrus and so on")

[{'start': 52,
  'end': 73,
  'text': 'left precentral gyrus',
  'label': 'Precentral_L'}]

bio-tags

In [14]:
bio_tags = [
 'O', 
 'B-BRAND', 'I-BRAND',
 'B-DOSAGE', 'I-DOSAGE',
 'B-GENERIC', 'I-GENERIC',
 'B-MODEL', 'I-MODEL'
]

dataset loading

In [ ]:
# Assuming you have your data in CoNLL format
def read_conll_file(file_path):
    sentences, labels = [], []
    current_sentence, current_labels = [], []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                word, label = line.split('\t')
                current_sentence.append(word)
                current_labels.append(label)
            else:
                if current_sentence:
                    sentences.append(current_sentence)
                    labels.append(current_labels)
                    current_sentence, current_labels = [], []
    
    if current_sentence:  # Don't forget the last sentence
        sentences.append(current_sentence)
        labels.append(current_labels)
    
    return sentences, labels

train_sentences, train_labels = read_conll_file('../../ner_train.csv')
dev_sentences, dev_labels = read_conll_file('../../ner_dev.csv')
test_sentences, test_labels = read_conll_file('../../ner_test.csv')

In [ ]:
def convert_to_dataset(sentences, labels, bio_tags):
    label_map = {tag: i for i, tag in enumerate(bio_tags)}
    
    # Convert string labels to IDs
    label_ids = [[label_map[label] for label in sentence_labels] for sentence_labels in labels]
    
    return Dataset.from_dict({
        "tokens": sentences,
        "ner_tags": label_ids
    }, features=Features({
        "tokens": [Value("string")],
        "ner_tags": [ClassLabel(names=bio_tags)]
    }))

train_dataset = convert_to_dataset(train_sentences, train_labels, bio_tags)
dev_dataset = convert_to_dataset(dev_sentences, dev_labels, bio_tags)
test_dataset = convert_to_dataset(test_sentences, test_labels, bio_tags)